# PDFCleaner Algorithm Prototype & Robustness Validation

This notebook demonstrates the client-side scanned document cleaning pipeline developed for **PDFCleaner** using Python and OpenCV. This serves as the algorithm prototype before porting the pipeline to TypeScript and OpenCV.js WASM.

## 10 Test Cases (5 Basic Scan Anomalies + 5 Composite Real-World Errors):
1. **Book Page (Shadows)**: Radial shadow from center camera angle.
2. **Receipt (Faded)**: Faded thermal receipt paper + salt & pepper noise.
3. **Journal (Yellow/Coffee)**: Aged yellow background with coffee stains.
4. **Photocopy (Heavy Noise)**: Black scanner borders + high density noise.
5. **Skewed Page (Tilted)**: Document tilted by 6.5 degrees.
6. **Comp 1: Shadows + Skew + Blur + GhostText**: 8-degree tilt + heavy radial shadow + out-of-focus blur + bleed-through ghost text.
7. **Comp 2: Jpeg + Stains + Creases + Staple**: JPEG artifacts + coffee stain + page fold creases + staple wire.
8. **Comp 3: Low DPI + Heavy Noise + Holes**: Pixelated low resolution + salt & pepper noise + punch holes.
9. **Comp 4: Charts + Signature + Stamps (Color)**: Color preservation test (blue signature + red stamp + green/red graph).
10. **Comp 5: Watermark + Pink Paper + Handwriting**: Background watermark text + pink paper tint + blue handwriting + finger occlusion.

## Processing Steps:
- **Anomaly Detection**: Run light checks for blur, overexposure, and margin cut-off.
- **Auto-Deskew**: Automatically detects text skew angle and rotates it straight.
- **Grayscale vs Color LAB Path**:
  - *Grayscale (Binarize)*: Convert to 1-channel grayscale -> Blur -> Background Norm -> Gamma -> Contrast Stretch -> Adaptive Threshold -> Morphological Open.
  - *Color (LAB space)*: Split LAB -> Process L channel only (Blur -> Background Norm -> Gamma -> Contrast Stretch) -> Merge L, A, B -> Convert back to BGR. Keeps stamps/signatures intact.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 10]
plt.rcParams['image.cmap'] = 'gray'

## 1. Load Sample Scans
Let's load the generated sample images from the `samples/` directory.

In [ ]:
sample_dir = 'samples'
samples = [
    'sample_book.png', 'sample_receipt.png', 'sample_handwriting.png', 'sample_photocopy.png', 'sample_skewed.png',
    'sample_composite_1.png', 'sample_composite_2.png', 'sample_composite_3.png', 'sample_composite_4.png', 'sample_composite_5.png'
]

for filename in samples:
    path = os.path.join(sample_dir, filename)
    if os.path.exists(path):
        print(f"Loaded {filename}: size {os.path.getsize(path)} bytes")
    else:
        print(f"Warning: {path} not found! Run generate_samples.py first.")

## 2. Document Cleaner Pipeline with Anomaly Detection & Color-Preservation

In [ ]:
class DocumentCleanerPipeline:
    def __init__(self, config):
        self.config = config
        self.timings = {}
        self.detected_skew_angle = 0.0
        self.warnings = []
        
    def detect_anomalies(self, bgr_img):
        self.warnings = []
        gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY) if len(bgr_img.shape) == 3 else bgr_img.copy()
        
        # 1. Blur Detection (Laplacian Variance)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        is_blurred = lap_var < 80.0
        if is_blurred:
            self.warnings.append(f"Blurry/Out of focus detected (Laplacian variance: {lap_var:.2f} < 80.0)")
            
        # 2. Overexposure Detection
        saturated_ratio = np.sum(gray >= 254) / gray.size
        is_overexposed = saturated_ratio > 0.25
        if is_overexposed:
            self.warnings.append(f"Overexposure/Glare detected ({saturated_ratio*100:.1f}% pixels are saturated white)")
            
        # 3. Cut-off margin text check
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        h, w = thresh.shape
        margin = 15
        top_pixels = np.sum(thresh[0:margin, :] > 0)
        bottom_pixels = np.sum(thresh[h-margin:h, :] > 0)
        left_pixels = np.sum(thresh[:, 0:margin] > 0)
        right_pixels = np.sum(thresh[:, w-margin:w] > 0)
        
        is_cutoff = (top_pixels > 50 or bottom_pixels > 50 or left_pixels > 50 or right_pixels > 50)
        if is_cutoff:
            self.warnings.append(f"Text touches edges. Border/Margins artifact (Top={top_pixels}, Bottom={bottom_pixels}, Left={left_pixels}, Right={right_pixels})")
            
        return {
            'blur_variance': lap_var,
            'saturated_ratio': saturated_ratio,
            'edge_pixels': (top_pixels, bottom_pixels, left_pixels, right_pixels)
        }

    def deskew(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
        thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        coords = np.column_stack(np.where(thresh > 0))
        if len(coords) == 0:
            return img.copy(), 0.0
        rect = cv2.minAreaRect(coords)
        angle = rect[-1]
        if angle < -45:
            angle = -(90 + angle)
        elif angle > 45:
            angle = 90 - angle
        else:
            angle = -angle
        (h, w) = img.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
        return rotated, angle

    def run(self, bgr_img):
        self.timings = {}
        stages = {}
        stages['original'] = bgr_img.copy()
        
        t_detect = time.perf_counter()
        self.detect_anomalies(bgr_img)
        self.timings['anomaly_detection'] = (time.perf_counter() - t_detect) * 1000
        
        current_img = bgr_img.copy()
        
        # 0. Deskew
        t0 = time.perf_counter()
        if self.config.get('enable_deskew', False):
            current_img, angle = self.deskew(current_img)
            self.detected_skew_angle = angle
        self.timings['deskew'] = (time.perf_counter() - t0) * 1000
        stages['deskewed'] = current_img.copy()
        
        grayscale = self.config.get('grayscale', True)
        if grayscale:
            # GRayscale Path
            t0 = time.perf_counter()
            gray = cv2.cvtColor(current_img, cv2.COLOR_BGR2GRAY)
            self.timings['grayscale'] = (time.perf_counter() - t0) * 1000
            stages['grayscale'] = gray.copy()
            
            # Blur
            t0 = time.perf_counter()
            if self.config.get('enable_noise_reduction', True):
                ksize = self.config.get('blur_kernel_size', 3)
                ksize = ksize if ksize % 2 == 1 else ksize + 1
                if self.config.get('blur_type', 'median') == 'median':
                    blurred = cv2.medianBlur(gray, ksize)
                else:
                    blurred = cv2.GaussianBlur(gray, (ksize, ksize), 0)
            else:
                blurred = gray.copy()
            self.timings['noise_reduction'] = (time.perf_counter() - t0) * 1000
            stages['noise_reduction'] = blurred.copy()
            
            # Bg Norm
            t0 = time.perf_counter()
            if self.config.get('enable_background_norm', True):
                norm_ksize = self.config.get('norm_kernel_size', 25)
                kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (norm_ksize, norm_ksize))
                bg = cv2.morphologyEx(blurred, cv2.MORPH_CLOSE, kernel)
                normalized = cv2.divide(blurred, bg, scale=255)
            else:
                normalized = blurred.copy()
            self.timings['background_normalization'] = (time.perf_counter() - t0) * 1000
            stages['background_normalization'] = normalized.copy()
            
            # Gamma
            t0 = time.perf_counter()
            gamma = self.config.get('gamma', 1.0)
            if gamma != 1.0:
                invGamma = 1.0 / gamma
                table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
                gamma_corrected = cv2.LUT(normalized, table)
            else:
                gamma_corrected = normalized.copy()
            self.timings['gamma_correction'] = (time.perf_counter() - t0) * 1000
            stages['gamma_correction'] = gamma_corrected.copy()
            
            # Contrast
            t0 = time.perf_counter()
            contrast_stretched = cv2.normalize(gamma_corrected, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
            self.timings['contrast_stretching'] = (time.perf_counter() - t0) * 1000
            stages['contrast_stretching'] = contrast_stretched.copy()
            
            # Threshold
            t0 = time.perf_counter()
            if self.config.get('enable_thresholding', True):
                block = self.config.get('threshold_block_size', 21)
                block = block if block % 2 == 1 else block + 1
                c_val = self.config.get('threshold_c', 10)
                thresholded = cv2.adaptiveThreshold(contrast_stretched, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block, c_val)
            else:
                thresholded = contrast_stretched.copy()
            self.timings['thresholding'] = (time.perf_counter() - t0) * 1000
            stages['thresholded'] = thresholded.copy()
            
            # Morphology
            t0 = time.perf_counter()
            if self.config.get('enable_morphology', True) and self.config.get('enable_thresholding', True):
                morph_size = self.config.get('morphology_kernel_size', 2)
                if morph_size > 0:
                    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (morph_size, morph_size))
                    cleaned = cv2.morphologyEx(thresholded, cv2.MORPH_OPEN, kernel)
                else:
                    cleaned = thresholded.copy()
            else:
                cleaned = thresholded.copy()
            self.timings['morphology'] = (time.perf_counter() - t0) * 1000
            stages['morphology'] = cleaned.copy()
            stages['final_output'] = cleaned.copy()
            
        else:
            # COLOR PATH (LAB SPACE)
            t0 = time.perf_counter()
            lab = cv2.cvtColor(current_img, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            self.timings['color_space_lab_convert'] = (time.perf_counter() - t0) * 1000
            stages['grayscale'] = l.copy()
            
            # Blur L channel
            t0 = time.perf_counter()
            if self.config.get('enable_noise_reduction', True):
                ksize = self.config.get('blur_kernel_size', 3)
                ksize = ksize if ksize % 2 == 1 else ksize + 1
                blurred_l = cv2.GaussianBlur(l, (ksize, ksize), 0)
            else:
                blurred_l = l.copy()
            self.timings['noise_reduction'] = (time.perf_counter() - t0) * 1000
            stages['noise_reduction'] = blurred_l.copy()
            
            # Bg Norm L channel
            t0 = time.perf_counter()
            if self.config.get('enable_background_norm', True):
                norm_ksize = self.config.get('norm_kernel_size', 35)
                kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (norm_ksize, norm_ksize))
                bg = cv2.morphologyEx(blurred_l, cv2.MORPH_CLOSE, kernel)
                normalized_l = cv2.divide(blurred_l, bg, scale=255)
            else:
                normalized_l = blurred_l.copy()
            self.timings['background_normalization'] = (time.perf_counter() - t0) * 1000
            stages['background_normalization'] = normalized_l.copy()
            
            # Gamma L channel
            t0 = time.perf_counter()
            gamma = self.config.get('gamma', 1.0)
            if gamma != 1.0:
                invGamma = 1.0 / gamma
                table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
                gamma_corrected_l = cv2.LUT(normalized_l, table)
            else:
                gamma_corrected_l = normalized_l.copy()
            self.timings['gamma_correction'] = (time.perf_counter() - t0) * 1000
            stages['gamma_correction'] = gamma_corrected_l.copy()
            
            # Contrast L channel
            t0 = time.perf_counter()
            contrast_stretched_l = cv2.normalize(gamma_corrected_l, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
            self.timings['contrast_stretching'] = (time.perf_counter() - t0) * 1000
            stages['contrast_stretching'] = contrast_stretched_l.copy()
            
            stages['thresholded'] = contrast_stretched_l.copy()
            stages['morphology'] = contrast_stretched_l.copy()
            
            # Merge and convert back to BGR
            t0 = time.perf_counter()
            merged = cv2.merge([contrast_stretched_l, a, b])
            final_bgr = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
            self.timings['color_space_bgr_convert'] = (time.perf_counter() - t0) * 1000
            stages['final_output'] = final_bgr.copy()
            
        return stages

## 3. Define Presets

In [ ]:
MODE_CONFIGS = {
    'light-clean': {
        'grayscale': False,
        'enable_noise_reduction': True,
        'blur_type': 'gaussian',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 35,
        'gamma': 1.1,
        'contrast': 1.1,
        'enable_thresholding': False,
        'enable_morphology': False
    },
    'strong-background-removal': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 25,
        'gamma': 0.8,
        'contrast': 1.4,
        'enable_thresholding': True,
        'threshold_block_size': 25,
        'threshold_c': 15,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    },
    'text-contrast-boost': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 21,
        'gamma': 0.6,
        'contrast': 1.6,
        'enable_thresholding': True,
        'threshold_block_size': 21,
        'threshold_c': 12,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    },
    'print-optimized': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 21,
        'gamma': 0.8,
        'contrast': 1.4,
        'enable_thresholding': True,
        'threshold_block_size': 21,
        'threshold_c': 15,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    },
    'compressed-output': {
        'grayscale': True,
        'enable_noise_reduction': True,
        'blur_type': 'median',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 15,
        'gamma': 0.9,
        'contrast': 1.3,
        'enable_thresholding': True,
        'threshold_block_size': 15,
        'threshold_c': 10,
        'enable_morphology': True,
        'morphology_kernel_size': 2
    },
    'color-preservation': {
        'grayscale': False,
        'enable_noise_reduction': True,
        'blur_type': 'gaussian',
        'blur_kernel_size': 3,
        'enable_background_norm': True,
        'norm_kernel_size': 31,
        'gamma': 0.9,
        'contrast': 1.3,
        'enable_thresholding': False,
        'enable_morphology': False
    }
}

## 4. Run Benchmark on 10 Test Cases

In [ ]:
test_cases = [
    ('Book Page (Shadows)', 'sample_book.png', 'strong-background-removal', False),
    ('Receipt (Faded)', 'sample_receipt.png', 'text-contrast-boost', False),
    ('Journal (Yellow/Coffee)', 'sample_handwriting.png', 'light-clean', False),
    ('Photocopy (Heavy Noise)', 'sample_photocopy.png', 'print-optimized', False),
    ('Skewed Page (Tilted)', 'sample_skewed.png', 'strong-background-removal', True),
    ('Comp 1: Shadows+Skew+Blur+GhostText', 'sample_composite_1.png', 'strong-background-removal', True),
    ('Comp 2: Jpeg+Stains+Creases+Staple', 'sample_composite_2.png', 'print-optimized', False),
    ('Comp 3: Low DPI+Heavy Noise+Holes', 'sample_composite_3.png', 'print-optimized', False),
    ('Comp 4: Charts+Signature+Stamps', 'sample_composite_4.png', 'color-preservation', False),
    ('Comp 5: Watermark+Pink+Handwriting', 'sample_composite_5.png', 'light-clean', False)
]

for title, filename, mode, enable_deskew in test_cases:
    img_path = os.path.join(sample_dir, filename)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Could not load {filename}")
        continue
        
    config = MODE_CONFIGS[mode].copy()
    config['enable_deskew'] = enable_deskew
    
    cleaner = DocumentCleanerPipeline(config)
    stages = cleaner.run(img)
    
    print(f"\n=== {title} ===")
    print(f"  Mode: {mode} | Deskew: {enable_deskew}")
    if enable_deskew:
        print(f"  Detected Skew Angle: {cleaner.detected_skew_angle:.2f} deg")
    if cleaner.warnings:
        print("  Warnings detected:")
        for w in cleaner.warnings:
            print(f"    {w}")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    cleaned_img = stages['final_output']
    if len(cleaned_img.shape) == 3:
        axes[1].imshow(cv2.cvtColor(cleaned_img, cv2.COLOR_BGR2RGB))
    else:
        axes[1].imshow(cleaned_img, cmap='gray')
    axes[1].set_title("Cleaned")
    axes[1].axis('off')
    
    out_dir = 'outputs'
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, f"compare_{filename}"), bbox_inches='tight')
    plt.close(fig) # prevent showing all inside notebook inline to save memory
    print(f"  Total processing time: {sum(cleaner.timings.values()):.2f} ms")
    print(f"  Saved cleaned image as outputs/cleaned_{filename}")

## 5. Visualizing Step-by-Step for Composite Case

In [ ]:
comp_img = cv2.imread(os.path.join(sample_dir, 'sample_composite_1.png'))
config = MODE_CONFIGS['strong-background-removal'].copy()
config['enable_deskew'] = True

cleaner = DocumentCleanerPipeline(config)
stages = cleaner.run(comp_img)

visual_stages = [
    ('original', 'Original BGR'),
    ('deskewed', 'Deskewed (Rotated)'),
    ('grayscale', 'Grayscale'),
    ('noise_reduction', 'Noise Reduction (Blur)'),
    ('background_normalization', 'Background Normalized'),
    ('gamma_correction', 'Gamma Corrected'),
    ('contrast_stretching', 'Contrast Stretched'),
    ('thresholded', 'Adaptive Thresholded'),
    ('morphology', 'Morphological Cleaned')
]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.ravel()

for idx, (key, title) in enumerate(visual_stages):
    stage_img = stages[key]
    if len(stage_img.shape) == 3:
        axes[idx].imshow(cv2.cvtColor(stage_img, cv2.COLOR_BGR2RGB))
    else:
        axes[idx].imshow(stage_img, cmap='gray')
    axes[idx].set_title(title)
    axes[idx].axis('off')

fig.savefig('outputs/composite_stages_visualization.png', bbox_inches='tight')
plt.show()

## 6. Key Learnings for Porting to OpenCV.js WASM

### Grayscale and Thresholding:
- Ensure correct channel orders. HTML5 Canvas coordinates are RGBA (so color space convert is `COLOR_RGBA2GRAY` instead of `COLOR_BGR2GRAY`).

### Color Preservation (LAB Path):
- Since we split the channels into L, A, B, we must manage their lifecycles. In OpenCV.js, splitting a matrix requires creating a vector of Mats:
  ```javascript
  let lab = new cv.Mat();
  cv.cvtColor(src, lab, cv.COLOR_RGBA2RGB); // Canvas -> RGBA -> RGB first
  cv.cvtColor(lab, lab, cv.COLOR_RGB2Lab);
  let planes = new cv.MatVector();
  cv.split(lab, planes);
  let l = planes.get(0);
  let a = planes.get(1);
  let b = planes.get(2);
  // Process L channel ...
  planes.set(0, processed_l);
  cv.merge(planes, lab);
  cv.cvtColor(lab, dst, cv.COLOR_Lab2RGB);
  cv.cvtColor(dst, dst, cv.COLOR_RGB2RGBA); // Convert back to canvas RGBA
  // Clean up all Mats in planes vector!
  l.delete(); a.delete(); b.delete(); planes.delete(); lab.delete();
  ```

### Manual Memory Management:
- Memory leaks are fatal in the Web Worker environment. Every single `new cv.Mat()` and any intermediate split output MUST be cleaned up using `delete()` inside `try/finally` blocks!